# Hyperparameter Optimisation

Tune only a small defensible candidate set within expanding training folds; do not inspect the final holdout during selection.


In [1]:
# PROJECT_ROOT_BOOTSTRAP: support kernels started from the repository root or notebooks/.
from pathlib import Path
import os
import sys

ROOT = Path.cwd()
if not (ROOT / "src").is_dir():
    ROOT = ROOT.parent
if not (ROOT / "src").is_dir():
    raise FileNotFoundError("Could not locate the project root containing src/.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

import json
from pathlib import Path
from importlib.metadata import version
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor
from src.data import load_demand, TARGETS
from src.features import make_calendar_features, make_lag_features
from src.backtesting import recursive_predict
from src.evaluation import expanding_splits, forecast_metrics

data = load_demand("data/raw/clientes_restaurantes.csv")
data["Total demand"] = data[list(TARGETS)].sum(axis=1, min_count=2)
development = data.iloc[:-28].copy()  # Keep the final 28 calendar dates untouched.
parameter_grid = [
    {"max_leaf_nodes": leaves, "learning_rate": rate}
    for leaves in (7, 15)
    for rate in (0.05, 0.1)
]
fold_rows = []
for target in [*TARGETS, "Total demand"]:
    y = development[target]
    splits = list(expanding_splits(len(y), initial=280, horizon=28, step=28))
    for parameters in parameter_grid:
        for fold, (tr, te) in enumerate(splits, start=1):
            train, test = y.iloc[tr].copy(), y.iloc[te].copy()
            if test.isna().any():
                print(target, fold, "excluded; unknown validation dates:", test.index[test.isna()].tolist())
                continue
            missing_dates = train.index[train.isna()]
            if len(missing_dates):
                print(target, fold, "training-only interpolation dates:", missing_dates.tolist())
            train = train.interpolate(limit_area="inside").dropna()
            X = make_calendar_features(train.index).join(make_lag_features(train))
            valid = X.notna().all(axis=1)
            model = HistGradientBoostingRegressor(random_state=42, **parameters)
            model.fit(X.loc[valid], train.loc[valid])
            def build_features(values):
                return make_calendar_features(values.index).join(make_lag_features(values))
            predicted = recursive_predict(model, train, test.index, build_features, target)
            fold_rows.append({"restaurant": target, "fold": fold, **parameters, **forecast_metrics(test, predicted, train)})
fold_results = pd.DataFrame(fold_rows)
summary = fold_results.groupby(["restaurant", "max_leaf_nodes", "learning_rate"]).agg(mae_mean=("mae", "mean"), mae_std=("mae", "std"), folds=("fold", "nunique")).reset_index()
display(summary)
selected = {}
for target in [*TARGETS, "Total demand"]:
    best = summary.loc[summary["restaurant"] == target].sort_values("mae_mean").iloc[0]
    selected[target] = {"max_leaf_nodes": int(best["max_leaf_nodes"]), "learning_rate": float(best["learning_rate"])}
Path("reports").mkdir(exist_ok=True)
Path("artifacts/models").mkdir(parents=True, exist_ok=True)
fold_results.to_csv("reports/ml_tuning_results.csv", index=False)
Path("artifacts/models/ml_hyperparameter_selection.json").write_text(json.dumps({
    "training_cutoff": str(development.index.max().date()),
    "validation_method": "recursive expanding-window development CV; final 28 calendar dates excluded",
    "forecast_horizon_days": 28,
    "candidate_grid": parameter_grid,
    "selected_parameters": selected,
    "package_versions": {"scikit-learn": version("scikit-learn"), "pandas": version("pandas")}
}, indent=2))
print("selected development parameters:", selected)


Chimi & Churri 1 training-only interpolation dates: [Timestamp('2016-04-05 00:00:00'), Timestamp('2016-09-17 00:00:00')]


Chimi & Churri 2 training-only interpolation dates: [Timestamp('2016-04-05 00:00:00'), Timestamp('2016-09-17 00:00:00')]


Chimi & Churri 3 training-only interpolation dates: [Timestamp('2016-04-05 00:00:00'), Timestamp('2016-09-17 00:00:00')]


Chimi & Churri 4 training-only interpolation dates: [Timestamp('2016-04-05 00:00:00'), Timestamp('2016-09-17 00:00:00')]


Chimi & Churri 5 training-only interpolation dates: [Timestamp('2016-04-05 00:00:00'), Timestamp('2016-09-17 00:00:00')]


Chimi & Churri 6 training-only interpolation dates: [Timestamp('2016-04-05 00:00:00'), Timestamp('2016-09-17 00:00:00')]


... (52 outputs left out)

Total demand 5 training-only interpolation dates: [Timestamp('2016-04-05 00:00:00'), Timestamp('2016-09-17 00:00:00'), Timestamp('2016-11-24 00:00:00')]


Total demand 6 training-only interpolation dates: [Timestamp('2016-04-05 00:00:00'), Timestamp('2016-09-17 00:00:00'), Timestamp('2016-11-24 00:00:00')]


,restaurant,max_leaf_nodes,learning_rate,mae_mean,mae_std,folds
0,Assa Frão,7,0.05,14.792147,5.426746,5
1,Assa Frão,7,0.10,15.393396,5.184149,5
2,Assa Frão,15,0.05,14.781985,4.105710,5
3,Assa Frão,15,0.10,15.036187,3.260124,5
4,Chimi & Churri,7,0.05,12.455973,2.311099,6
5,Chimi & Churri,7,0.10,12.587621,2.338362,6
6,Chimi & Churri,15,0.05,12.508333,1.911665,6
7,Chimi & Churri,15,0.10,12.975732,2.346503,6
8,Total demand,7,0.05,22.148339,6.238060,5
9,Total demand,7,0.10,22.883087,6.979214,5


selected development parameters: {'Chimi & Churri': {'max_leaf_nodes': 7, 'learning_rate': 0.05}, 'Assa Frão': {'max_leaf_nodes': 15, 'learning_rate': 0.05}, 'Total demand': {'max_leaf_nodes': 7, 'learning_rate': 0.05}}
